In [1]:
# ================================================================
# TASK 6 - PYTHON DATA ANALYSIS
# Growth Instrumentation & North-Star Metrics
# ================================================================

import pandas as pd
import numpy as np
from IPython.display import display
from google.colab import files

# ================================================================
# 1. LOAD CLEANED DATASET
# ================================================================

file_name = "Cleaned_Dataset.xlsx"

df = pd.read_excel(file_name)

print("=" * 70)
print("TASK 6 - PYTHON ANALYSIS")
print("=" * 70)

print("\nDataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())


# ================================================================
# 2. BASIC DATA OVERVIEW
# ================================================================

print("\n" + "=" * 70)
print("1. DATASET OVERVIEW")
print("=" * 70)

print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

print("\nColumn Names:")
for col in df.columns:
    print("-", col)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
display(df.isnull().sum().to_frame("Missing_Values"))


# ================================================================
# 3. DUPLICATE CHECK
# ================================================================

print("\n" + "=" * 70)
print("2. DUPLICATE CHECK")
print("=" * 70)

duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)


# ================================================================
# 4. DESCRIPTIVE STATISTICS
# ================================================================

print("\n" + "=" * 70)
print("3. DESCRIPTIVE STATISTICS")
print("=" * 70)

display(df.describe())


# ================================================================
# 5. IDENTIFY IMPORTANT NUMERIC COLUMNS
# ================================================================

numeric_columns = [
    "sessions",
    "signups",
    "activated_users",
    "paid_users",
    "revenue",
    "support_tickets",
    "customer_satisfaction"
]

numeric_columns = [
    col for col in numeric_columns
    if col in df.columns
]

print("\nNumeric Analysis Columns:")
print(numeric_columns)


# ================================================================
# 6. CORE BUSINESS METRICS
# ================================================================

print("\n" + "=" * 70)
print("4. CORE BUSINESS METRICS")
print("=" * 70)

total_sessions = df["sessions"].sum()
total_signups = df["signups"].sum()
total_activated = df["activated_users"].sum()
total_paid = df["paid_users"].sum()

total_revenue = df["revenue"].sum()
average_revenue = df["revenue"].mean()

total_support_tickets = df["support_tickets"].sum()
average_satisfaction = df["customer_satisfaction"].mean()

print(f"Total Sessions              : {total_sessions:,.0f}")
print(f"Total Signups               : {total_signups:,.0f}")
print(f"Total Activated Users       : {total_activated:,.0f}")
print(f"Total Paid Users            : {total_paid:,.0f}")
print(f"Total Revenue               : {total_revenue:,.2f}")
print(f"Average Revenue             : {average_revenue:,.2f}")
print(f"Total Support Tickets       : {total_support_tickets:,.0f}")
print(f"Average Customer Satisfaction: {average_satisfaction:.2f}")


# ================================================================
# 7. CONVERSION METRICS
# ================================================================

print("\n" + "=" * 70)
print("5. CONVERSION METRICS")
print("=" * 70)

def safe_rate(numerator, denominator):
    if denominator == 0:
        return 0
    return (numerator / denominator) * 100


signup_rate = safe_rate(
    total_signups,
    total_sessions
)

activation_rate = safe_rate(
    total_activated,
    total_signups
)

paid_conversion_rate = safe_rate(
    total_paid,
    total_activated
)

overall_paid_conversion = safe_rate(
    total_paid,
    total_sessions
)

print(f"Session → Signup Rate          : {signup_rate:.2f}%")
print(f"Signup → Activation Rate       : {activation_rate:.2f}%")
print(f"Activation → Paid User Rate    : {paid_conversion_rate:.2f}%")
print(f"Session → Paid User Rate       : {overall_paid_conversion:.2f}%")


# ================================================================
# 8. NORTH-STAR METRIC ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("6. NORTH-STAR METRIC ANALYSIS")
print("=" * 70)

print("""
Candidate North-Star Metrics:

1. Paid Users
2. Revenue
3. Activated Users

The analysis will examine the relationship between
the user journey and these business outcomes.
""")

# Candidate metrics
north_star_candidates = pd.DataFrame({
    "Metric": [
        "Paid Users",
        "Revenue",
        "Activated Users"
    ],
    "Value": [
        total_paid,
        total_revenue,
        total_activated
    ]
})

display(north_star_candidates)


# ================================================================
# 9. NORTH-STAR INPUT FUNNEL
# ================================================================

print("\n" + "=" * 70)
print("7. NORTH-STAR INPUT FUNNEL")
print("=" * 70)

funnel = pd.DataFrame({
    "Stage": [
        "Sessions",
        "Signups",
        "Activated Users",
        "Paid Users"
    ],
    "Value": [
        total_sessions,
        total_signups,
        total_activated,
        total_paid
    ]
})

funnel["Conversion_From_Previous"] = [
    np.nan,
    signup_rate,
    activation_rate,
    paid_conversion_rate
]

display(funnel)


# ================================================================
# 10. INPUT DRIVER ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("8. INPUT DRIVER ANALYSIS")
print("=" * 70)

driver_columns = [
    "sessions",
    "signups",
    "activated_users",
    "revenue",
    "support_tickets",
    "customer_satisfaction"
]

driver_columns = [
    col for col in driver_columns
    if col in df.columns
]

correlation_table = (
    df[driver_columns + ["paid_users"]]
    .corr()["paid_users"]
    .sort_values(ascending=False)
    .to_frame("Correlation_with_Paid_Users")
)

display(correlation_table)


# ================================================================
# 11. CORRELATION WITH REVENUE
# ================================================================

print("\n" + "=" * 70)
print("9. DRIVERS OF REVENUE")
print("=" * 70)

revenue_driver_columns = [
    "sessions",
    "signups",
    "activated_users",
    "paid_users",
    "support_tickets",
    "customer_satisfaction"
]

revenue_driver_columns = [
    col for col in revenue_driver_columns
    if col in df.columns
]

revenue_correlation = (
    df[revenue_driver_columns + ["revenue"]]
    .corr()["revenue"]
    .sort_values(ascending=False)
    .to_frame("Correlation_with_Revenue")
)

display(revenue_correlation)


# ================================================================
# 12. CHANNEL ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("10. ACQUISITION CHANNEL ANALYSIS")
print("=" * 70)

if "acquisition_channel" in df.columns:

    channel_analysis = (
        df.groupby("acquisition_channel")
        .agg(
            Sessions=("sessions", "sum"),
            Signups=("signups", "sum"),
            Activated_Users=("activated_users", "sum"),
            Paid_Users=("paid_users", "sum"),
            Revenue=("revenue", "sum"),
            Support_Tickets=("support_tickets", "sum"),
            Avg_Satisfaction=("customer_satisfaction", "mean")
        )
        .reset_index()
    )

    channel_analysis["Signup_Rate_%"] = (
        channel_analysis["Signups"] /
        channel_analysis["Sessions"] * 100
    )

    channel_analysis["Activation_Rate_%"] = (
        channel_analysis["Activated_Users"] /
        channel_analysis["Signups"].replace(0, np.nan) * 100
    )

    channel_analysis["Paid_Conversion_%"] = (
        channel_analysis["Paid_Users"] /
        channel_analysis["Activated_Users"].replace(0, np.nan) * 100
    )

    channel_analysis = channel_analysis.fillna(0)

    channel_analysis = channel_analysis.sort_values(
        "Paid_Users",
        ascending=False
    )

    display(channel_analysis)


# ================================================================
# 13. PRODUCT ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("11. PRODUCT ANALYSIS")
print("=" * 70)

if "product" in df.columns:

    product_analysis = (
        df.groupby("product")
        .agg(
            Sessions=("sessions", "sum"),
            Signups=("signups", "sum"),
            Activated_Users=("activated_users", "sum"),
            Paid_Users=("paid_users", "sum"),
            Revenue=("revenue", "sum"),
            Support_Tickets=("support_tickets", "sum"),
            Avg_Satisfaction=("customer_satisfaction", "mean")
        )
        .reset_index()
    )

    product_analysis["Signup_Rate_%"] = (
        product_analysis["Signups"] /
        product_analysis["Sessions"].replace(0, np.nan) * 100
    )

    product_analysis["Activation_Rate_%"] = (
        product_analysis["Activated_Users"] /
        product_analysis["Signups"].replace(0, np.nan) * 100
    )

    product_analysis["Paid_Conversion_%"] = (
        product_analysis["Paid_Users"] /
        product_analysis["Activated_Users"].replace(0, np.nan) * 100
    )

    product_analysis = product_analysis.fillna(0)

    product_analysis = product_analysis.sort_values(
        "Paid_Users",
        ascending=False
    )

    display(product_analysis)


# ================================================================
# 14. REGION ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("12. REGIONAL ANALYSIS")
print("=" * 70)

if "region" in df.columns:

    region_analysis = (
        df.groupby("region")
        .agg(
            Sessions=("sessions", "sum"),
            Signups=("signups", "sum"),
            Activated_Users=("activated_users", "sum"),
            Paid_Users=("paid_users", "sum"),
            Revenue=("revenue", "sum"),
            Support_Tickets=("support_tickets", "sum"),
            Avg_Satisfaction=("customer_satisfaction", "mean")
        )
        .reset_index()
    )

    region_analysis["Signup_Rate_%"] = (
        region_analysis["Signups"] /
        region_analysis["Sessions"].replace(0, np.nan) * 100
    )

    region_analysis["Activation_Rate_%"] = (
        region_analysis["Activated_Users"] /
        region_analysis["Signups"].replace(0, np.nan) * 100
    )

    region_analysis["Paid_Conversion_%"] = (
        region_analysis["Paid_Users"] /
        region_analysis["Activated_Users"].replace(0, np.nan) * 100
    )

    region_analysis = region_analysis.fillna(0)

    region_analysis = region_analysis.sort_values(
        "Paid_Users",
        ascending=False
    )

    display(region_analysis)


# ================================================================
# 15. TIME ANALYSIS
# ================================================================

print("\n" + "=" * 70)
print("13. TIME-BASED ANALYSIS")
print("=" * 70)

if "event_date" in df.columns:

    monthly_analysis = (
        df.groupby(
            pd.Grouper(
                key="event_date",
                freq="ME"
            )
        )
        .agg(
            Sessions=("sessions", "sum"),
            Signups=("signups", "sum"),
            Activated_Users=("activated_users", "sum"),
            Paid_Users=("paid_users", "sum"),
            Revenue=("revenue", "sum"),
            Support_Tickets=("support_tickets", "sum"),
            Avg_Satisfaction=("customer_satisfaction", "mean")
        )
        .reset_index()
    )

    monthly_analysis["Signup_Rate_%"] = (
        monthly_analysis["Signups"] /
        monthly_analysis["Sessions"].replace(0, np.nan) * 100
    )

    monthly_analysis["Activation_Rate_%"] = (
        monthly_analysis["Activated_Users"] /
        monthly_analysis["Signups"].replace(0, np.nan) * 100
    )

    monthly_analysis["Paid_Conversion_%"] = (
        monthly_analysis["Paid_Users"] /
        monthly_analysis["Activated_Users"].replace(0, np.nan) * 100
    )

    monthly_analysis = monthly_analysis.fillna(0)

    display(monthly_analysis)


# ================================================================
# 16. GUARDRAIL METRICS
# ================================================================

print("\n" + "=" * 70)
print("14. GUARDRAIL METRICS")
print("=" * 70)

guardrails = pd.DataFrame({
    "Guardrail_Metric": [
        "Customer Satisfaction",
        "Support Tickets",
        "Activation Rate",
        "Paid Conversion Rate"
    ],
    "Current_Value": [
        average_satisfaction,
        total_support_tickets,
        activation_rate,
        paid_conversion_rate
    ]
})

display(guardrails)

print("""
Interpretation:

Customer Satisfaction
→ Should remain healthy while growth increases.

Support Tickets
→ Should not increase excessively as user activity grows.

Activation Rate
→ Should remain stable or improve.

Paid Conversion Rate
→ Should remain stable or improve while pursuing growth.
""")


# ================================================================
# 17. GROWTH MODEL
# ================================================================

print("\n" + "=" * 70)
print("15. GROWTH MODEL")
print("=" * 70)

growth_model = pd.DataFrame({
    "Growth_Stage": [
        "Traffic / Sessions",
        "Signups",
        "Activation",
        "Paid Conversion",
        "North-Star Outcome"
    ],
    "Metric": [
        "Sessions",
        "Signups",
        "Activated Users",
        "Paid Users",
        "Paid Users / Revenue"
    ],
    "Relationship": [
        "Sessions create acquisition opportunity",
        "Sessions convert into Signups",
        "Signups convert into Activated Users",
        "Activated Users convert into Paid Users",
        "Paid Users contribute to Revenue"
    ]
})

display(growth_model)


# ================================================================
# 18. GROWTH LEVER CALCULATION
# ================================================================

print("\n" + "=" * 70)
print("16. GROWTH LEVER SIMULATION")
print("=" * 70)

# Simple scenario analysis:
# What happens to paid users if conversion improves?

current_paid_users = total_paid

scenario_rates = [
    1.05,
    1.10,
    1.15,
    1.20
]

growth_scenarios = []

for multiplier in scenario_rates:

    projected_paid = total_paid * multiplier

    growth_scenarios.append({
        "Scenario": f"{int((multiplier - 1) * 100)}% Increase",
        "Current_Paid_Users": total_paid,
        "Projected_Paid_Users": projected_paid,
        "Additional_Paid_Users": projected_paid - total_paid
    })

growth_scenarios = pd.DataFrame(growth_scenarios)

display(growth_scenarios)


# ================================================================
# 19. TOP PERFORMERS
# ================================================================

print("\n" + "=" * 70)
print("17. TOP PERFORMERS")
print("=" * 70)

if "acquisition_channel" in df.columns:

    top_channel = (
        channel_analysis
        .sort_values("Paid_Users", ascending=False)
        .iloc[0]
    )

    print(
        "Top Acquisition Channel by Paid Users:",
        top_channel["acquisition_channel"]
    )

    print(
        "Paid Users:",
        top_channel["Paid_Users"]
    )

    print(
        "Revenue:",
        round(top_channel["Revenue"], 2)
    )


if "product" in df.columns:

    top_product = (
        product_analysis
        .sort_values("Paid_Users", ascending=False)
        .iloc[0]
    )

    print(
        "\nTop Product by Paid Users:",
        top_product["product"]
    )

    print(
        "Paid Users:",
        top_product["Paid_Users"]
    )

    print(
        "Revenue:",
        round(top_product["Revenue"], 2)
    )


if "region" in df.columns:

    top_region = (
        region_analysis
        .sort_values("Paid_Users", ascending=False)
        .iloc[0]
    )

    print(
        "\nTop Region by Paid Users:",
        top_region["region"]
    )

    print(
        "Paid Users:",
        top_region["Paid_Users"]
    )

    print(
        "Revenue:",
        round(top_region["Revenue"], 2)
    )


# ================================================================
# 20. AUTOMATIC INSIGHTS
# ================================================================

print("\n" + "=" * 70)
print("18. KEY BUSINESS INSIGHTS")
print("=" * 70)

insights = []

# Funnel insight
if signup_rate > 0:
    insights.append(
        f"1. The overall Session-to-Signup conversion rate is "
        f"{signup_rate:.2f}%."
    )

if activation_rate > 0:
    insights.append(
        f"2. The Signup-to-Activation conversion rate is "
        f"{activation_rate:.2f}%."
    )

if paid_conversion_rate > 0:
    insights.append(
        f"3. The Activation-to-Paid conversion rate is "
        f"{paid_conversion_rate:.2f}%."
    )

# Driver insight
if "paid_users" in correlation_table.index:

    driver_corr = (
        correlation_table
        .drop(index="paid_users")
        .iloc[:, 0]
    )

    strongest_driver = driver_corr.abs().idxmax()

    strongest_value = driver_corr[strongest_driver]

    insights.append(
        f"4. Among the analyzed numeric variables, "
        f"{strongest_driver} has the strongest absolute "
        f"correlation with Paid Users "
        f"({strongest_value:.3f})."
    )

# Satisfaction insight
insights.append(
    f"5. Average Customer Satisfaction is "
    f"{average_satisfaction:.2f} out of 5."
)

# Revenue insight
insights.append(
    f"6. Total Revenue generated in the dataset is "
    f"{total_revenue:,.2f}."
)

for insight in insights:
    print(insight)


# ================================================================
# 21. EXPORT ANALYSIS RESULTS TO EXCEL
# ================================================================

print("\n" + "=" * 70)
print("19. EXPORTING ANALYSIS RESULTS")
print("=" * 70)

analysis_file = "Task_6_Analysis_Results.xlsx"

with pd.ExcelWriter(analysis_file, engine="openpyxl") as writer:

    df.to_excel(
        writer,
        sheet_name="Cleaned_Data",
        index=False
    )

    north_star_candidates.to_excel(
        writer,
        sheet_name="North_Star",
        index=False
    )

    funnel.to_excel(
        writer,
        sheet_name="Funnel",
        index=False
    )

    correlation_table.to_excel(
        writer,
        sheet_name="Paid_User_Drivers"
    )

    revenue_correlation.to_excel(
        writer,
        sheet_name="Revenue_Drivers"
    )

    guardrails.to_excel(
        writer,
        sheet_name="Guardrails",
        index=False
    )

    growth_model.to_excel(
        writer,
        sheet_name="Growth_Model",
        index=False
    )

    growth_scenarios.to_excel(
        writer,
        sheet_name="Growth_Scenarios",
        index=False
    )

    if "acquisition_channel" in df.columns:
        channel_analysis.to_excel(
            writer,
            sheet_name="Channel_Analysis",
            index=False
        )

    if "product" in df.columns:
        product_analysis.to_excel(
            writer,
            sheet_name="Product_Analysis",
            index=False
        )

    if "region" in df.columns:
        region_analysis.to_excel(
            writer,
            sheet_name="Region_Analysis",
            index=False
        )

    if "event_date" in df.columns:
        monthly_analysis.to_excel(
            writer,
            sheet_name="Monthly_Analysis",
            index=False
        )

print("Analysis Excel created successfully.")


# ================================================================
# 22. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("FINAL ANALYSIS SUMMARY")
print("=" * 70)

print(f"""
Total Sessions          : {total_sessions:,.0f}
Total Signups           : {total_signups:,.0f}
Total Activated Users   : {total_activated:,.0f}
Total Paid Users        : {total_paid:,.0f}

Total Revenue           : {total_revenue:,.2f}

Signup Rate             : {signup_rate:.2f}%
Activation Rate         : {activation_rate:.2f}%
Paid Conversion Rate    : {paid_conversion_rate:.2f}%

Average Satisfaction    : {average_satisfaction:.2f}/5
Total Support Tickets   : {total_support_tickets:,.0f}
""")

print("=" * 70)
print("PYTHON ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 70)

TASK 6 - PYTHON ANALYSIS

Dataset loaded successfully!
Rows: 500
Columns: 16


,customer_id,event_date,product,acquisition_channel,region,sessions,signups,activated_users,paid_users,revenue,support_tickets,customer_satisfaction,year,month,month_name,quarter
0,CUST071,2025-07-15 11:52:47.134,Basic Plan,Direct,North,6,0,1,2,159.22,2,3.1,2025,7,July,Q3
1,CUST006,2025-02-23 06:00:43.287,Standard Plan,Social Media,North,6,4,0,2,480.87,5,4.4,2025,2,February,Q1
2,CUST002,2025-07-30 19:31:37.395,Premium Plan,Social Media,North,7,4,0,1,396.20,4,3.2,2025,7,July,Q3
3,CUST042,2025-04-24 01:35:13.828,Standard Plan,Social Media,West,10,2,3,0,471.58,4,4.8,2025,4,April,Q2
4,CUST029,2025-03-17 20:43:46.052,Basic Plan,Social Media,North,12,1,1,0,210.96,3,4.6,2025,3,March,Q1



1. DATASET OVERVIEW
Number of Rows: 500
Number of Columns: 16

Column Names:
- customer_id
- event_date
- product
- acquisition_channel
- region
- sessions
- signups
- activated_users
- paid_users
- revenue
- support_tickets
- customer_satisfaction
- year
- month
- month_name
- quarter

Data Types:
customer_id                      object
event_date               datetime64[ns]
product                          object
acquisition_channel              object
region                           object
sessions                          int64
signups                           int64
activated_users                   int64
paid_users                        int64
revenue                         float64
support_tickets                   int64
customer_satisfaction           float64
year                              int64
month                             int64
month_name                       object
quarter                          object
dtype: object

Missing Values:


,Missing_Values
customer_id,0
event_date,0
product,0
acquisition_channel,0
region,0
sessions,0
signups,0
activated_users,0
paid_users,0
revenue,0



2. DUPLICATE CHECK
Duplicate Rows: 0

3. DESCRIPTIVE STATISTICS


...


Interpretation:

Customer Satisfaction
→ Should remain healthy while growth increases.

Support Tickets
→ Should not increase excessively as user activity grows.

Activation Rate
→ Should remain stable or improve.

Paid Conversion Rate
→ Should remain stable or improve while pursuing growth.


15. GROWTH MODEL


,Growth_Stage,Metric,Relationship
0,Traffic / Sessions,Sessions,Sessions create acquisition opportunity
1,Signups,Signups,Sessions convert into Signups
2,Activation,Activated Users,Signups convert into Activated Users
3,Paid Conversion,Paid Users,Activated Users convert into Paid Users
4,North-Star Outcome,Paid Users / Revenue,Paid Users contribute to Revenue



16. GROWTH LEVER SIMULATION


,Scenario,Current_Paid_Users,Projected_Paid_Users,Additional_Paid_Users
0,5% Increase,490,514.5,24.5
1,10% Increase,490,539.0,49.0
2,14% Increase,490,563.5,73.5
3,19% Increase,490,588.0,98.0



17. TOP PERFORMERS
Top Acquisition Channel by Paid Users: Direct
Paid Users: 114
Revenue: 25469.66

Top Product by Paid Users: Standard Plan
Paid Users: 177
Revenue: 46558.74

Top Region by Paid Users: South
Paid Users: 128
Revenue: 35809.63

18. KEY BUSINESS INSIGHTS
1. The overall Session-to-Signup conversion rate is 27.55%.
2. The Signup-to-Activation conversion rate is 72.39%.
3. The Activation-to-Paid conversion rate is 66.04%.
4. Among the analyzed numeric variables, support_tickets has the strongest absolute correlation with Paid Users (0.074).
5. Average Customer Satisfaction is 3.71 out of 5.
6. Total Revenue generated in the dataset is 130,738.62.

19. EXPORTING ANALYSIS RESULTS
Analysis Excel created successfully.

FINAL ANALYSIS SUMMARY

Total Sessions          : 3,721
Total Signups           : 1,025
Total Activated Users   : 742
Total Paid Users        : 490

Total Revenue           : 130,738.62

Signup Rate             : 27.55%
Activation Rate         : 72.39%
Paid Conve